In [1]:
!pip install /kaggle/input/pip-install-lifelines/autograd-1.7.0-py3-none-any.whl
!pip install /kaggle/input/pip-install-lifelines/autograd-gamma-0.5.0.tar.gz
!pip install /kaggle/input/pip-install-lifelines/interface_meta-1.3.0-py3-none-any.whl
!pip install /kaggle/input/pip-install-lifelines/formulaic-1.0.2-py3-none-any.whl
!pip install /kaggle/input/pip-install-lifelines/lifelines-0.30.0-py3-none-any.whl

Processing /kaggle/input/pip-install-lifelines/autograd-1.7.0-py3-none-any.whl
Processing /kaggle/input/pip-install-lifelines/autograd-gamma-0.5.0.tar.gz
  Preparing metadata (setup.py) ... - done
  Created wheel for autograd-gamma: filename=autograd_gamma-0.5.0-py3-none-any.whl size=4030 sha256=de165c47c9ee9df0d80aa08292451acccde38edac05289960358e231cb3e0726
  Stored in directory: /root/.cache/pip/wheels/6b/b5/e0/4c79e15c0b5f2c15ecf613c720bb20daab20a666eb67135155
Successfully built autograd-gamma
Processing /kaggle/input/pip-install-lifelines/interface_meta-1.3.0-py3-none-any.whl
Processing /kaggle/input/pip-install-lifelines/formulaic-1.0.2-py3-none-any.whl
Processing /kaggle/input/pip-install-lifelines/lifelines-0.30.0-py3-none-any.whl


In [2]:
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

In [3]:
import numpy as np
import polars as pl
import pandas as pd
import plotly.colors as pc
import plotly.express as px
import plotly.graph_objects as go

In [4]:
import plotly.io as pio
pio.renderers.default = 'iframe'

In [5]:
pd.options.display.max_columns = None

In [6]:
import lightgbm as lgb
from metric import score
from scipy.stats import rankdata 
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from lifelines import CoxPHFitter, KaplanMeierFitter, NelsonAalenFitter

In [7]:
class CFG:

    train_path = Path('/kaggle/input/equity-post-HCT-survival-predictions/train.csv')
    test_path = Path('/kaggle/input/equity-post-HCT-survival-predictions/test.csv')
    subm_path = Path('/kaggle/input/equity-post-HCT-survival-predictions/sample_submission.csv')

    colorscale = 'Sunset'
    color = '#EADDCA'

    batch_size = 32768
    early_stop = 300
    penalizer = 0.01
    n_splits = 5

    ctb_params = {
        'bootstrap_type': 'Bernoulli',
        'loss_function': 'RMSE',
        'learning_rate': 0.03,
        'num_trees': 10000,
        'random_state': 42,
        'task_type': 'CPU',
        'subsample': 0.85,
        'reg_lambda': 8.0,
        'depth': 8
    }

    lgb_params = {
        'objective': 'regression',
        'min_child_samples': 32,
        'num_iterations': 10000,
        'learning_rate': 0.03,
        'extra_trees': True,
        'reg_lambda': 8.0,
        'reg_alpha': 0.1,
        'num_leaves': 64,
        'metric': 'rmse',
        'max_depth': 8,
        'device': 'cpu',
        'max_bin': 128,
        'verbose': -1,
        'seed': 42
    }

In [8]:
class FE:

    def __init__(self, batch_size):
        self.batch_size = batch_size

    def load_data(self, path):

        return pl.read_csv(path, batch_size=self.batch_size)

    def cast_datatypes(self, df):

        num_cols = [
            'hla_high_res_8',
            'hla_low_res_8',
            'hla_high_res_6',
            'hla_low_res_6',
            'hla_high_res_10',
            'hla_low_res_10',
            'hla_match_dqb1_high',
            'hla_match_dqb1_low',
            'hla_match_drb1_high',
            'hla_match_drb1_low',
            'hla_nmdp_6',
            'year_hct',
            'hla_match_a_high',
            'hla_match_a_low',
            'hla_match_b_high',
            'hla_match_b_low',
            'hla_match_c_high',
            'hla_match_c_low',
            'donor_age',
            'age_at_hct',
            'comorbidity_score',
            'karnofsky_score',
            'efs',
            'efs_time'
        ]

        for col in df.columns:

            if col in num_cols:
                df = df.with_columns(pl.col(col).fill_null(-1).cast(pl.Float32))  

            else:
                df = df.with_columns(pl.col(col).fill_null('Unknown').cast(pl.String))  

        return df.with_columns(pl.col('ID').cast(pl.Int32))

    def info(self, df):
        
        print(f'\nShape of dataframe: {df.shape}') 
        
        mem = df.memory_usage().sum() / 1024**2
        print('Memory usage: {:.2f} MB\n'.format(mem))

        display(df.head())

    def apply_fe(self, path):

        df = self.load_data(path)
        df = self.cast_datatypes(df)
        df = df.to_pandas()

        self.info(df)
        
        cat_cols = [col for col in df.columns if df[col].dtype == pl.String]

        return df, cat_cols

In [9]:
fe = FE(CFG.batch_size)

In [10]:
train_data, cat_cols = fe.apply_fe(CFG.train_path)


Shape of dataframe: (28800, 60)
Memory usage: 10.44 MB



,ID,dri_score,psych_disturb,cyto_score,diabetes,hla_match_c_high,hla_high_res_8,tbi_status,arrhythmia,hla_low_res_6,graft_type,vent_hist,renal_issue,pulm_severe,prim_disease_hct,hla_high_res_6,cmv_status,hla_high_res_10,hla_match_dqb1_high,tce_imm_match,hla_nmdp_6,hla_match_c_low,rituximab,hla_match_drb1_low,hla_match_dqb1_low,prod_type,cyto_score_detail,conditioning_intensity,ethnicity,year_hct,obesity,mrd_hct,in_vivo_tcd,tce_match,hla_match_a_high,hepatic_severe,donor_age,prior_tumor,hla_match_b_low,peptic_ulcer,age_at_hct,hla_match_a_low,gvhd_proph,rheum_issue,sex_match,hla_match_b_high,race_group,comorbidity_score,karnofsky_score,hepatic_mild,tce_div_match,donor_related,melphalan_dose,hla_low_res_8,cardiac,hla_match_drb1_high,pulm_moderate,hla_low_res_10,efs,efs_time
0,0,N/A - non-malignant indication,No,Unknown,No,-1.0,-1.0,No TBI,No,6.0,Bone marrow,No,No,No,IEA,6.0,+/+,-1.0,2.0,Unknown,6.0,2.0,No,2.0,2.0,BM,Unknown,Unknown,Not Hispanic or Latino,2016.0,No,Unknown,Yes,Unknown,2.0,No,-1.000000,No,2.0,No,9.942000,2.0,FKalone,No,M-F,2.0,More than one race,0.0,90.0,No,Unknown,Unrelated,"N/A, Mel not given",8.0,No,2.0,No,10.0,0.0,42.355999
1,1,Intermediate,No,Intermediate,No,2.0,8.0,"TBI +- Other, >cGy",No,6.0,Peripheral blood,No,No,No,AML,6.0,+/+,10.0,2.0,P/P,6.0,2.0,No,2.0,2.0,PB,Intermediate,MAC,Not Hispanic or Latino,2008.0,No,Positive,No,Permissive,2.0,No,72.290001,No,2.0,No,43.705002,2.0,Other GVHD Prophylaxis,No,F-F,2.0,Asian,3.0,90.0,No,Permissive mismatched,Related,"N/A, Mel not given",8.0,No,2.0,Yes,10.0,1.0,4.672000
2,2,N/A - non-malignant indication,No,Unknown,No,2.0,8.0,No TBI,No,6.0,Bone marrow,No,No,No,HIS,6.0,+/+,10.0,2.0,P/P,6.0,2.0,No,2.0,2.0,BM,Unknown,Unknown,Not Hispanic or Latino,2019.0,No,Unknown,Yes,Unknown,2.0,No,-1.000000,No,2.0,No,33.997002,2.0,Cyclophosphamide alone,No,F-M,2.0,More than one race,0.0,90.0,No,Permissive mismatched,Related,"N/A, Mel not given",8.0,No,2.0,No,10.0,0.0,19.792999
3,3,High,No,Intermediate,No,2.0,8.0,No TBI,No,6.0,Bone marrow,No,No,No,ALL,6.0,+/+,10.0,2.0,P/P,6.0,2.0,No,2.0,2.0,BM,Intermediate,MAC,Not Hispanic or Latino,2009.0,No,Positive,No,Permissive,2.0,No,29.230000,No,2.0,No,43.244999,2.0,FK+ MMF +- others,No,M-M,2.0,White,0.0,90.0,Yes,Permissive mismatched,Unrelated,"N/A, Mel not given",8.0,No,2.0,No,10.0,0.0,102.348999
4,4,High,No,Unknown,No,2.0,8.0,No TBI,No,6.0,Peripheral blood,No,No,No,MPN,6.0,+/+,10.0,2.0,Unknown,5.0,2.0,No,2.0,2.0,PB,Unknown,MAC,Hispanic or Latino,2018.0,No,Unknown,Yes,Unknown,2.0,No,56.810001,No,2.0,No,29.740000,2.0,TDEPLETION +- other,No,M-F,2.0,American Indian or Alaska Native,1.0,90.0,No,Permissive mismatched,Related,MEL,8.0,No,2.0,No,10.0,0.0,16.223000


In [11]:
test_data, _ = fe.apply_fe(CFG.test_path)


Shape of dataframe: (3, 58)
Memory usage: 0.00 MB



,ID,dri_score,psych_disturb,cyto_score,diabetes,hla_match_c_high,hla_high_res_8,tbi_status,arrhythmia,hla_low_res_6,graft_type,vent_hist,renal_issue,pulm_severe,prim_disease_hct,hla_high_res_6,cmv_status,hla_high_res_10,hla_match_dqb1_high,tce_imm_match,hla_nmdp_6,hla_match_c_low,rituximab,hla_match_drb1_low,hla_match_dqb1_low,prod_type,cyto_score_detail,conditioning_intensity,ethnicity,year_hct,obesity,mrd_hct,in_vivo_tcd,tce_match,hla_match_a_high,hepatic_severe,donor_age,prior_tumor,hla_match_b_low,peptic_ulcer,age_at_hct,hla_match_a_low,gvhd_proph,rheum_issue,sex_match,hla_match_b_high,race_group,comorbidity_score,karnofsky_score,hepatic_mild,tce_div_match,donor_related,melphalan_dose,hla_low_res_8,cardiac,hla_match_drb1_high,pulm_moderate,hla_low_res_10
0,28800,N/A - non-malignant indication,No,Unknown,No,-1.0,-1.0,No TBI,No,6.0,Bone marrow,No,No,No,IEA,6.0,+/+,-1.0,2.0,Unknown,6.0,2.0,No,2.0,2.0,BM,Unknown,Unknown,Not Hispanic or Latino,2016.0,No,Unknown,Yes,Unknown,2.0,No,-1.000000,No,2.0,No,9.942000,2.0,FKalone,No,M-F,2.0,More than one race,0.0,90.0,No,Unknown,Unrelated,"N/A, Mel not given",8.0,No,2.0,No,10.0
1,28801,Intermediate,No,Intermediate,No,2.0,8.0,"TBI +- Other, >cGy",No,6.0,Peripheral blood,No,No,No,AML,6.0,+/+,10.0,2.0,P/P,6.0,2.0,No,2.0,2.0,PB,Intermediate,MAC,Not Hispanic or Latino,2008.0,No,Positive,No,Permissive,2.0,No,72.290001,No,2.0,No,43.705002,2.0,Other GVHD Prophylaxis,No,F-F,2.0,Asian,3.0,90.0,No,Permissive mismatched,Related,"N/A, Mel not given",8.0,No,2.0,Yes,10.0
2,28802,N/A - non-malignant indication,No,Unknown,No,2.0,8.0,No TBI,No,6.0,Bone marrow,No,No,No,HIS,6.0,+/+,10.0,2.0,P/P,6.0,2.0,No,2.0,2.0,BM,Unknown,Unknown,Not Hispanic or Latino,2019.0,No,Unknown,Yes,Unknown,2.0,No,-1.000000,No,2.0,No,33.997002,2.0,Cyclophosphamide alone,No,F-M,2.0,More than one race,0.0,90.0,No,Permissive mismatched,Related,"N/A, Mel not given",8.0,No,2.0,No,10.0


In [12]:
class EDA:
    
    def __init__(self, colorscale, color, df):
        self.colorscale = colorscale
        self.color = color  
        self.df = df  

    def template(self, fig, title):
        
        fig.update_layout(
            title=title,
            title_x=0.5, 
            plot_bgcolor='rgba(40, 40, 43, 1)',  
            paper_bgcolor='rgba(40, 40, 43, 1)', 
            font=dict(color=self.color),
            margin=dict(l=72, r=72, t=72, b=72), 
            height=720
        )
        
        return fig

    def distribution_plot(self, col, title):
        
        fig = px.histogram(
            self.df,
            x=col,
            nbins=100,
            color_discrete_sequence=[self.color]
        )
        
        fig.update_layout(
            xaxis_title='Values',
            yaxis_title='Count',
            bargap=0.1,
            xaxis=dict(gridcolor='grey'),
            yaxis=dict(gridcolor='grey', zerolinecolor='grey')
        )
        
        fig.update_traces(hovertemplate='Value: %{x:.2f}<br>Count: %{y:,}')
        
        fig = self.template(fig, f'{title}')
        fig.show()

    def bubble_chart(self, col, title):
        
        value_counts = self.df[col].value_counts().reset_index()
        value_counts.columns = [col, 'count']
        
        fig = px.scatter(
            value_counts,
            x=col,
            y='count',
            size='count',
            color='count',
            size_max=240,
            color_continuous_scale=self.colorscale,
            labels={col: col.capitalize(), 'count': 'Count'},
            hover_name=col,
            hover_data={'count': True}
        )
        
        fig.update_layout(
            title_text=f'{title}',
            title_x=0.5,
            plot_bgcolor='rgba(40, 40, 43, 1)',
            paper_bgcolor='rgba(40, 40, 43, 1)',
            font=dict(color=self.color),
            margin=dict(l=20, r=20, t=50, b=20),
            xaxis_title='Values',
            yaxis_title='Count',
            height=720
        )
        
        fig.update_traces(
            marker=dict(line=dict(width=1, color='DarkSlateGrey')),
            hovertemplate=(
                '<b>Value:</b> %{x}<br>'
                '<b>Count:</b> %{y:,}<br>'
            ),
            hoverlabel=dict(
                font=dict(color=self.color), 
                bgcolor='rgba(40, 40, 43, 1)'
            )
        )
        
        fig = self.template(fig, f'{title}')
        fig.show()

    def pie_chart(self, col, title):
        
        value_counts = self.df[col].value_counts().reset_index()
        value_counts.columns = [col, 'count']
        
        fig = px.pie(
            value_counts,
            names=col,
            values='count',
            color=col,
            color_discrete_sequence=px.colors.sequential.Sunset,
            hole=0.2
        )
        
        fig.update_layout(
            title_x=0.5,
            plot_bgcolor='rgba(40, 40, 43, 1)',
            paper_bgcolor='rgba(40, 40, 43, 1)',
            font=dict(color=self.color),
            margin=dict(l=20, r=20, t=50, b=20),
            height=720
        )
        
        fig.update_traces(
            textinfo='percent+label',
            hovertemplate=(
                '<b>Value:</b> %{label}<br>'
                '<b>Count:</b> %{value:,}<br>'
                '<b>Percentage:</b> %{percent}<br>'
            ),
            hoverlabel=dict(
                font=dict(color=self.color),
                bgcolor='rgba(40, 40, 43, 1)'
            )
        )
        
        fig = self.template(fig, f'{title}')
        fig.show()

    def bar_chart(self, col, title):
        
        value_counts = self.df[col].value_counts().reset_index()
        value_counts.columns = [col, 'count']
        
        fig = px.bar(
            value_counts,
            x=col,
            y='count',
            color='count',
            color_continuous_scale=self.colorscale,
            labels={col: col.capitalize(), 'count': 'Count'},
        )
        
        fig.update_layout(
            xaxis_title='Values',
            yaxis_title='Count',
            bargap=0.2,
            xaxis=dict(gridcolor='grey'),
            yaxis=dict(gridcolor='grey', zerolinecolor='grey'),
            plot_bgcolor='rgba(40, 40, 43, 1)',
            paper_bgcolor='rgba(40, 40, 43, 1)',
            font=dict(color=self.color),
            margin=dict(l=20, r=20, t=50, b=20),
            height=720
        )
        
        fig.update_traces(
            hovertemplate=(
                '<b>Value:</b> %{x}<br>'
                '<b>Count:</b> %{y:,}<br>'
            ),
            marker=dict(line=dict(width=1, color='DarkSlateGrey')),
            hoverlabel=dict(
                font=dict(color=self.color), 
                bgcolor='rgba(40, 40, 43, 1)'
            )
        )
        
        fig = self.template(fig, f'{title}')
        fig.show()

In [13]:
eda = EDA(CFG.colorscale, CFG.color, train_data)

In [14]:
class MD:
    
    def __init__(self, early_stop, penalizer, n_splits, color):
        self.early_stop = early_stop
        self.penalizer = penalizer
        self.n_splits = n_splits
        self.color = color
        
    def _plot_cv(self, fold_scores, title, metric='C-Index'):
        
        fold_scores = [round(score, 3) for score in fold_scores]
        mean_score = round(np.mean(fold_scores), 3)

        fig = go.Figure()

        fig.add_trace(go.Scatter(
            x = list(range(1, len(fold_scores) + 1)),
            y = fold_scores,
            mode = 'markers', 
            name = 'Fold Scores',
            marker = dict(size = 27, color=self.color, symbol='diamond'),
            text = [f'{score:.3f}' for score in fold_scores],
            hovertemplate = 'Fold %{x}: %{text}<extra></extra>',
            hoverlabel=dict(font=dict(size=18))  
        ))

        fig.add_trace(go.Scatter(
            x = [1, len(fold_scores)],
            y = [mean_score, mean_score],
            mode = 'lines',
            name = f'Mean: {mean_score:.3f}',
            line = dict(dash = 'dash', color = '#FFBF00'),
            hoverinfo = 'none'
        ))
        
        fig.update_layout(
            title = f'{title} | Cross-validation {metric} scores: {mean_score}',
            xaxis_title = 'Fold',
            yaxis_title = f'{metric} Score',
            plot_bgcolor = 'rgba(40, 40, 43, 1)',  
            paper_bgcolor = 'rgba(40, 40, 43, 1)',
            font = dict(color=self.color), 
            xaxis = dict(
                gridcolor = 'grey',
                tickmode = 'linear',
                tick0 = 1,
                dtick = 1,
                range = [0.5, len(fold_scores) + 0.5],
                zerolinecolor = 'grey'
            ),
            yaxis = dict(
                gridcolor = 'grey',
                zerolinecolor = 'grey'
            )
        )
        
        fig.show()

    def create_target1(self, data, cat_cols):
        
        cph_data = pd.get_dummies(data, columns=cat_cols, drop_first=True)
        
        cph = CoxPHFitter(penalizer=self.penalizer)
        cph.fit(cph_data, duration_col='efs_time', event_col='efs')
    
        data['target1'] = cph.predict_partial_hazard(cph_data)
        
        return data

    def create_target2(self, data):
        
        kmf = KaplanMeierFitter()
        kmf.fit(durations=data['efs_time'], event_observed=data['efs'])

        data['target2'] = kmf.survival_function_at_times(data['efs_time']).values

        return data

    def create_target3(self, data):
        
        naf = NelsonAalenFitter()
        naf.fit(durations=data['efs_time'], event_observed=data['efs'])
    
        data['target3'] = naf.cumulative_hazard_at_times(data['efs_time']).values
        data['target3'] = data['target3'] * -1
        
        return data
        
    def train_model(self, data, cat_cols, params, target, title):
        
        for col in cat_cols:
            data[col] = data[col].astype('category')
            
        X = data.drop(['ID', 'efs', 'efs_time', 'target1', 'target2', 'target3'], axis=1)
        y = data[target]
        
        models, fold_scores = [], []
            
        cv = KFold(n_splits=self.n_splits, shuffle=True, random_state=42)
                
        # Initialize out-of-fold predictions array
        oof_preds = np.zeros(len(X))
    
        for fold, (train_index, valid_index) in enumerate(cv.split(X, y)):
                
            X_train = X.iloc[train_index]
            X_valid = X.iloc[valid_index]
                
            y_train = y.iloc[train_index]
            y_valid = y.iloc[valid_index]
    
            if title.startswith('LightGBM'):
                        
                model = lgb.LGBMRegressor()
                        
                model.fit(
                    X_train, 
                    y_train,  
                    eval_set=[(X_valid, y_valid)],
                    eval_metric='rmse',
                    callbacks=[lgb.early_stopping(self.early_stop, verbose=0), lgb.log_evaluation(0)]
                )
                        
            elif title.startswith('CatBoost'):
                        
                model = CatBoostRegressor(verbose=0, cat_features=cat_cols)
                        
                model.fit(
                    X_train,
                    y_train,
                    eval_set=(X_valid, y_valid),
                    early_stopping_rounds=self.early_stop, 
                    verbose=0
                )               
                    
            models.append(model)
                
            oof_preds[valid_index] = model.predict(X_valid)

            y_true_fold = data.iloc[valid_index][['ID', 'efs', 'efs_time', 'race_group']].copy()
            y_pred_fold = data.iloc[valid_index][['ID']].copy()
            
            y_pred_fold['prediction'] = oof_preds[valid_index]
    
            fold_score = score(y_true_fold, y_pred_fold, 'ID')
            fold_scores.append(fold_score)
    
        self._plot_cv(fold_scores, title)
            
        y_true = data[['ID', 'efs', 'efs_time', 'race_group']].copy()
        y_pred = data[['ID']].copy()
        
        y_pred['prediction'] = oof_preds
            
        c_index_score = score(y_true.copy(), y_pred.copy(), 'ID')
        print(f'\nOverall C-Index for {title}: {c_index_score:.3f}\n')
        
        return models, oof_preds

    def infer_model(self, data, cat_cols, models):
        
        data = data.drop(['ID'], axis=1)

        for col in cat_cols:
            data[col] = data[col].astype('category')

        return np.mean([model.predict(data) for model in models], axis=0)

In [15]:
md = MD(CFG.early_stop, CFG.penalizer, CFG.n_splits, CFG.color)

In [16]:
train_data = md.create_target1(train_data, cat_cols)
eda.distribution_plot(col='target1', title='Cox Target') 

In [17]:
train_data = md.create_target2(train_data)
eda.distribution_plot(col='target2', title='Kaplan-Meier Target') 

In [18]:
train_data = md.create_target3(train_data)
eda.distribution_plot(col='target3', title='Nelson-Aalen Target') 

In [19]:
fe.info(train_data)


Shape of dataframe: (28800, 63)
Memory usage: 11.10 MB



,ID,dri_score,psych_disturb,cyto_score,diabetes,hla_match_c_high,hla_high_res_8,tbi_status,arrhythmia,hla_low_res_6,graft_type,vent_hist,renal_issue,pulm_severe,prim_disease_hct,hla_high_res_6,cmv_status,hla_high_res_10,hla_match_dqb1_high,tce_imm_match,hla_nmdp_6,hla_match_c_low,rituximab,hla_match_drb1_low,hla_match_dqb1_low,prod_type,cyto_score_detail,conditioning_intensity,ethnicity,year_hct,obesity,mrd_hct,in_vivo_tcd,tce_match,hla_match_a_high,hepatic_severe,donor_age,prior_tumor,hla_match_b_low,peptic_ulcer,age_at_hct,hla_match_a_low,gvhd_proph,rheum_issue,sex_match,hla_match_b_high,race_group,comorbidity_score,karnofsky_score,hepatic_mild,tce_div_match,donor_related,melphalan_dose,hla_low_res_8,cardiac,hla_match_drb1_high,pulm_moderate,hla_low_res_10,efs,efs_time,target1,target2,target3
0,0,N/A - non-malignant indication,No,Unknown,No,-1.0,-1.0,No TBI,No,6.0,Bone marrow,No,No,No,IEA,6.0,+/+,-1.0,2.0,Unknown,6.0,2.0,No,2.0,2.0,BM,Unknown,Unknown,Not Hispanic or Latino,2016.0,No,Unknown,Yes,Unknown,2.0,No,-1.000000,No,2.0,No,9.942000,2.0,FKalone,No,M-F,2.0,More than one race,0.0,90.0,No,Unknown,Unrelated,"N/A, Mel not given",8.0,No,2.0,No,10.0,0.0,42.355999,0.258357,0.458687,-0.779367
1,1,Intermediate,No,Intermediate,No,2.0,8.0,"TBI +- Other, >cGy",No,6.0,Peripheral blood,No,No,No,AML,6.0,+/+,10.0,2.0,P/P,6.0,2.0,No,2.0,2.0,PB,Intermediate,MAC,Not Hispanic or Latino,2008.0,No,Positive,No,Permissive,2.0,No,72.290001,No,2.0,No,43.705002,2.0,Other GVHD Prophylaxis,No,F-F,2.0,Asian,3.0,90.0,No,Permissive mismatched,Related,"N/A, Mel not given",8.0,No,2.0,Yes,10.0,1.0,4.672000,1.120131,0.847759,-0.165155
2,2,N/A - non-malignant indication,No,Unknown,No,2.0,8.0,No TBI,No,6.0,Bone marrow,No,No,No,HIS,6.0,+/+,10.0,2.0,P/P,6.0,2.0,No,2.0,2.0,BM,Unknown,Unknown,Not Hispanic or Latino,2019.0,No,Unknown,Yes,Unknown,2.0,No,-1.000000,No,2.0,No,33.997002,2.0,Cyclophosphamide alone,No,F-M,2.0,More than one race,0.0,90.0,No,Permissive mismatched,Related,"N/A, Mel not given",8.0,No,2.0,No,10.0,0.0,19.792999,0.142673,0.462424,-0.771252
3,3,High,No,Intermediate,No,2.0,8.0,No TBI,No,6.0,Bone marrow,No,No,No,ALL,6.0,+/+,10.0,2.0,P/P,6.0,2.0,No,2.0,2.0,BM,Intermediate,MAC,Not Hispanic or Latino,2009.0,No,Positive,No,Permissive,2.0,No,29.230000,No,2.0,No,43.244999,2.0,FK+ MMF +- others,No,M-M,2.0,White,0.0,90.0,Yes,Permissive mismatched,Unrelated,"N/A, Mel not given",8.0,No,2.0,No,10.0,0.0,102.348999,1.323702,0.456661,-0.783792
4,4,High,No,Unknown,No,2.0,8.0,No TBI,No,6.0,Peripheral blood,No,No,No,MPN,6.0,+/+,10.0,2.0,Unknown,5.0,2.0,No,2.0,2.0,PB,Unknown,MAC,Hispanic or Latino,2018.0,No,Unknown,Yes,Unknown,2.0,No,56.810001,No,2.0,No,29.740000,2.0,TDEPLETION +- other,No,M-F,2.0,American Indian or Alaska Native,1.0,90.0,No,Permissive mismatched,Related,MEL,8.0,No,2.0,No,10.0,0.0,16.223000,0.974314,0.464674,-0.766400


In [20]:
ctb1_models, _ = md.train_model(train_data, cat_cols, CFG.ctb_params, target='target1', title='CatBoost')


Overall C-Index for CatBoost: 0.660



In [21]:
lgb1_models, _ = md.train_model(train_data, cat_cols, CFG.lgb_params, target='target1', title='LightGBM')

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009590 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 842
[LightGBM] [Info] Number of data points in the train set: 23040, number of used features: 57
[LightGBM] [Info] Start training from score 1.256212
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010046 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 840
[LightGBM] [Info] Number of data points in the train set: 23040, number of used features: 57
[LightGBM] [Info] Start training from score 1.256542
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010232 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough,


Overall C-Index for LightGBM: 0.656



In [22]:
ctb1_preds = md.infer_model(test_data, cat_cols, ctb1_models)

In [23]:
lgb1_preds = md.infer_model(test_data, cat_cols, lgb1_models)

In [24]:
ctb2_models, _ = md.train_model(train_data, cat_cols, CFG.ctb_params, target='target2', title='CatBoost')


Overall C-Index for CatBoost: 0.674



In [25]:
lgb2_models, _ = md.train_model(train_data, cat_cols, CFG.lgb_params, target='target2', title='LightGBM')

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010171 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 842
[LightGBM] [Info] Number of data points in the train set: 23040, number of used features: 57
[LightGBM] [Info] Start training from score 0.606473
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009914 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 840
[LightGBM] [Info] Number of data points in the train set: 23040, number of used features: 57
[LightGBM] [Info] Start training from score 0.605167
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010068 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough,


Overall C-Index for LightGBM: 0.667



In [26]:
ctb2_preds = md.infer_model(test_data, cat_cols, ctb2_models)

In [27]:
lgb2_preds = md.infer_model(test_data, cat_cols, lgb2_models)

<p style="background-color: #EADDCA; font-size: 300%; text-align: center; border-radius: 40px 40px; color: #C4A484; font-weight: bold; font-family: 'Roboto'; border: 4px solid #C4A484; text-shadow: 2px 2px 0.5px rgba(0, 0, 0, 0.05);">Models with Nelson-Aalen Target</p>

In [28]:
ctb3_models, _ = md.train_model(train_data, cat_cols, CFG.ctb_params, target='target3', title='CatBoost')


Overall C-Index for CatBoost: 0.676



In [29]:
lgb3_models, _ = md.train_model(train_data, cat_cols, CFG.lgb_params, target='target3', title='LightGBM')

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010095 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 842
[LightGBM] [Info] Number of data points in the train set: 23040, number of used features: 57
[LightGBM] [Info] Start training from score -0.538910
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010164 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 840
[LightGBM] [Info] Number of data points in the train set: 23040, number of used features: 57
[LightGBM] [Info] Start training from score -0.540892
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010048 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enoug


Overall C-Index for LightGBM: 0.670



In [30]:
ctb3_preds = md.infer_model(test_data, cat_cols, ctb3_models)

In [31]:
lgb3_preds = md.infer_model(test_data, cat_cols, lgb3_models)

<p style="background-color: #EADDCA; font-size: 300%; text-align: center; border-radius: 40px 40px; color: #C4A484; font-weight: bold; font-family: 'Roboto'; border: 4px solid #C4A484; text-shadow: 2px 2px 0.5px rgba(0, 0, 0, 0.05);">Ensemble Model</p>

In [32]:
preds = [ctb1_preds, lgb1_preds, ctb2_preds, lgb2_preds, ctb3_preds, lgb3_preds]

In [33]:
ranked_preds = np.array([rankdata(p) for p in preds])

In [34]:
ensemble_preds = np.mean(ranked_preds, axis=0)

In [35]:
subm_data = pd.read_csv(CFG.subm_path)
subm_data['prediction'] = ensemble_preds

In [36]:
subm_data.to_csv('submission.csv', index=False)
display(subm_data.head())

,ID,prediction
0,28800,2.0
1,28801,3.0
2,28802,1.0
